# AI-Powered Crop Yield Prediction System
### End-to-End Predictive Analytics & Machine Learning Pipeline

**Objective:** Build a predictive model that forecasts agricultural crop yield based on environmental and agricultural variables such as rainfall, temperature, pesticides, region, and crop type.

---

## 1. Import Libraries
We start by importing the necessary libraries for data processing, visualization, modeling, and serialization.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Ensure plots display inline
%matplotlib inline
sns.set_theme(style="whitegrid")

## 2. Load Dataset
We load the agricultural crop dataset `yield_df.csv` using Pandas.

In [ ]:
# Path to the dataset
dataset_path = os.path.join("..", "data", "yield_df.csv")

# Check if exists, fall back if run from root directory
if not os.path.exists(dataset_path):
    dataset_path = os.path.join("data", "yield_df.csv")

df = pd.read_csv(dataset_path)
print("Dataset loaded successfully!")

## 3. Dataset Overview
Let's examine the structure, columns, types, and shape of our crop dataset.

In [ ]:
# Display dimension of dataset
print("Shape of dataset:", df.shape)

# Show first 5 rows
df.head()

In [ ]:
# Display columns info
df.info()

In [ ]:
# Statistical summary of the dataset
df.describe()

## 4. Data Cleaning
We clean up the dataset by dropping unnamed index columns and removing duplicate records.

In [ ]:
# Drop unnamed index columns if present
if df.columns[0].startswith("Unnamed") or df.columns[0] == "":
    df = df.iloc[:, 1:]

# Check for duplicated rows
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

if duplicates > 0:
    df = df.drop_duplicates()
    print("Duplicates removed. New shape:", df.shape)

## 5. Missing Values Check
Identify any missing or null values in columns to maintain data integrity.

In [ ]:
print("Null counts per column:")
df.isnull().sum()

## 6. Exploratory Data Analysis (EDA)
Visualizing statistical relationships and distributions of our variables.

## 7. Correlation Heatmap
Let's see the linear correlation between the numerical columns: Year, average rainfall, pesticide use, average temperature, and crop yield.

In [ ]:
plt.figure(figsize=(8, 6))
numerical_cols = ['Year', 'average_rain_fall_mm_per_year', 'pesticides_tonnes', 'avg_temp', 'hg/ha_yield']
corr_matrix = df[numerical_cols].corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix of Crop Variables', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../images/heatmap.png', dpi=300)
plt.show()

## 8. Yield Distribution
Visualize the skewness and distribution of the target crop yield variable `hg/ha_yield`.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['hg/ha_yield'], kde=True, color='#2ec4b6', bins=30)
plt.title('Distribution of Crop Yield (hg/ha)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Yield (hg/ha)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.tight_layout()
plt.savefig('../images/yield_distribution.png', dpi=300)
plt.show()

## 9. Rainfall vs Yield Plot
Analyze the correlation and interaction between average annual rainfall and crop yield.

In [ ]:
plt.figure(figsize=(10, 6))
# Sample data points to ensure scatter plot is readable
df_sample = df.sample(n=min(2000, len(df)), random_state=42)

sns.scatterplot(data=df_sample, x='average_rain_fall_mm_per_year', y='hg/ha_yield', alpha=0.5, color='#457b9d')
plt.title('Rainfall vs Crop Yield', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Average Rainfall (mm/year)', fontsize=12)
plt.ylabel('Yield (hg/ha)', fontsize=12)
plt.tight_layout()
plt.savefig('../images/rainfall_vs_yield.png', dpi=300)
plt.show()

## 10. Temperature vs Yield Plot
See how average yearly temperature impacts agricultural yields.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_sample, x='avg_temp', y='hg/ha_yield', alpha=0.5, color='#e63946')
plt.title('Temperature vs Crop Yield', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Average Temperature (°C)', fontsize=12)
plt.ylabel('Yield (hg/ha)', fontsize=12)
plt.tight_layout()
plt.savefig('../images/temperature_vs_yield.png', dpi=300)
plt.show()

## 11. Top Crop Producing Areas
Analyze which countries/areas produce the highest average crop yields in our dataset.

In [ ]:
plt.figure(figsize=(12, 6))
top_areas = df.groupby('Area')['hg/ha_yield'].mean().nlargest(10).reset_index()
sns.barplot(data=top_areas, x='hg/ha_yield', y='Area', palette='viridis')
plt.title('Top 10 Crop Producing Countries (Average Yield)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Average Yield (hg/ha)', fontsize=12)
plt.ylabel('Country', fontsize=12)
plt.tight_layout()
plt.savefig('../images/top_crops.png', dpi=300)
plt.show()

## 12. Feature Engineering
Prepare features and partition the data into descriptive attributes ($X$) and the prediction goal ($y$).

In [ ]:
X = df.drop(columns=['hg/ha_yield'])
y = df['hg/ha_yield']

print("Feature Columns:", list(X.columns))
print("Target Column: hg/ha_yield")

## 13. Encoding Categorical Variables
We apply `OneHotEncoder` to categorical fields (`Area` and `Item`) and standard scaling to numeric fields using a reusable Scikit-Learn pipeline.

In [ ]:
categorical_features = ['Area', 'Item']
numerical_features = ['Year', 'average_rain_fall_mm_per_year', 'pesticides_tonnes', 'avg_temp']

# Define column transformer for preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])
print("Preprocessor Pipeline initialized successfully!")

## 14. Train-Test Split
Split dataset into 80% training data and 20% validation test data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training instances: {X_train.shape[0]}")
print(f"Testing instances: {X_test.shape[0]}")

## 15. Random Forest Regressor
Build and train the primary model.

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', rf_model)])

print("Training Random Forest Regressor...")
rf_pipeline.fit(X_train, y_train)
print("Random Forest Training Completed!")

## 16. Linear Regression
Train a simple benchmark Linear Regression model.

In [ ]:
lr_model = LinearRegression()
lr_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', lr_model)])

print("Training Linear Regression...")
lr_pipeline.fit(X_train, y_train)
print("Linear Regression Training Completed!")

## 17. Decision Tree Regressor
Train a single decision tree model.

In [ ]:
dt_model = DecisionTreeRegressor(max_depth=15, random_state=42)
dt_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', dt_model)])

print("Training Decision Tree Regressor...")
dt_pipeline.fit(X_train, y_train)
print("Decision Tree Training Completed!")

## 18. Model Evaluation
Evaluate all three models on the test split using R² Score, MAE, and RMSE.

In [ ]:
pipelines = {
    'Linear Regression': lr_pipeline,
    'Decision Tree Regressor': dt_pipeline,
    'Random Forest Regressor': rf_pipeline
}

results = {}
for name, pipeline in pipelines.items():
    preds = pipeline.predict(X_test)
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    
    results[name] = {'R2': r2, 'MAE': mae, 'RMSE': rmse}
    
    print(f"=== {name} Metrics ===")
    print(f"  R² Score : {r2:.4f}")
    print(f"  MAE      : {mae:.2f}")
    print(f"  RMSE     : {rmse:.2f}\n")

## 19. R² Score Comparison
Visualizing R² scores side-by-side.

In [ ]:
r2_vals = [results[m]['R2'] for m in results]
plt.figure(figsize=(8, 5))
sns.barplot(x=list(results.keys()), y=r2_vals, palette='Blues_r')
plt.title('R² Score Comparison of Regression Models', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('R² Score', fontsize=12)
plt.ylim(0, 1.1)
for i, val in enumerate(r2_vals):
    plt.text(i, val + 0.02, f"{val:.4f}", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 20. MAE and RMSE Comparison
Show values in a formatted table.

In [ ]:
results_df = pd.DataFrame(results).T
results_df

## 21. Prediction Comparison Graph
Compare prediction lines against actual yields for the first 50 validation samples.

In [ ]:
plt.figure(figsize=(12, 6))
rf_preds = rf_pipeline.predict(X_test)
dt_preds = dt_pipeline.predict(X_test)
lr_preds = lr_pipeline.predict(X_test)

plt.plot(np.array(y_test)[:50], label='Actual Yield', color='black', linewidth=2.5, marker='o')
plt.plot(rf_preds[:50], label='Random Forest Preds', color='#2ec4b6', linestyle='--', marker='x')
plt.plot(dt_preds[:50], label='Decision Tree Preds', color='#ff9f1c', linestyle=':', marker='^')
plt.plot(lr_preds[:50], label='Linear Regression Preds', color='#e63946', linestyle='-.', marker='s')

plt.title('Actual vs Predicted Yield Comparison (First 50 Samples)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Sample Index', fontsize=12)
plt.ylabel('Yield (hg/ha)', fontsize=12)
plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../images/prediction_comparison.png', dpi=300)
plt.show()

## 22. Feature Importance Plot
Visualize relative node impurity importance for the Random Forest model features.

In [ ]:
cat_encoder = rf_pipeline.named_steps['preprocessor'].named_transformers_['cat']
one_hot_cols = cat_encoder.get_feature_names_out(categorical_features).tolist()
feature_names = numerical_features + one_hot_cols
importances = rf_pipeline.named_steps['regressor'].feature_importances_

indices = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices], y=np.array(feature_names)[indices], palette='mako')
plt.title('Top 15 Feature Importances (Random Forest Regressor)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Relative Importance Value', fontsize=12)
plt.ylabel('Features / Categories', fontsize=12)
plt.tight_layout()
plt.savefig('../images/feature_importance.png', dpi=300)
plt.show()

## 23. Prediction System
Build a custom interface function that accepts single-row inputs and outputs crop yield prediction.

In [ ]:
def predict_yield(area, item, year, rainfall, pesticides, temp):
    input_data = pd.DataFrame([{
        'Area': area,
        'Item': item,
        'Year': year,
        'average_rain_fall_mm_per_year': rainfall,
        'pesticides_tonnes': pesticides,
        'avg_temp': temp
    }])
    prediction = rf_pipeline.predict(input_data)[0]
    return prediction

# Sample prediction
sample_area = df['Area'].iloc[0]
sample_item = df['Item'].iloc[0]
sample_year = df['Year'].iloc[0]
sample_rain = df['average_rain_fall_mm_per_year'].iloc[0]
sample_pest = df['pesticides_tonnes'].iloc[0]
sample_temp = df['avg_temp'].iloc[0]

pred = predict_yield(sample_area, sample_item, sample_year, sample_rain, sample_pest, sample_temp)
print(f"Sample Input Features: {sample_area}, {sample_item}, {sample_year}, Rain: {sample_rain}mm, Pest: {sample_pest}t, Temp: {sample_temp}°C")
print(f"Predicted Yield : {pred:.2f} hg/ha (Actual: {df['hg/ha_yield'].iloc[0]} hg/ha)")

## 24. Model Saving using pickle
Serialize the trained Random Forest pipeline so it can be loaded in the Streamlit application.

In [ ]:
# Ensure models directory exists
os.makedirs('../models', exist_ok=True)

# Save primary model pipeline
with open('../models/random_forest.pkl', 'wb') as f:
    pickle.dump(rf_pipeline, f)

# Save unique areas and items for metadata
app_meta = {
    'areas': sorted(df['Area'].unique().tolist()),
    'items': sorted(df['Item'].unique().tolist()),
    'results': results
}

with open('../models/preprocessor.pkl', 'wb') as f:
    pickle.dump(app_meta, f)

print("Trained model pipeline and metadata successfully saved to models/")

## 25. Conclusion
Our model evaluation indicates that the **Random Forest Regressor** outperforms the alternative models by a significant margin. It explains a substantial percentage of yield variance, capturing interactions between crop types, temperature levels, and soil inputs. The exported model serves as the analytical backend for the interactive Streamlit agricultural planning dashboard.